# 🤖 Autonomous Data Analyst — BI-Backed Edition
### Analysis engine: **Apache Superset** (Apache 2.0) · Support/Explain layer: `phi4-mini-reasoning` + `qwen2.5-coder:7b-instruct` (via Ollama)



## Pipeline

1. **Data ingestion** — your CSV/Excel files are loaded and synced into a local DuckDB file.
   This file is registered as a **Database connection inside Superset**, so Superset is querying
   the real data, not a copy the LLM controls.
2. **Qwen2.5-Coder ("Hands")** — turns the natural-language question into **one read-only SQL
   query**. It no longer executes anything itself.
3. **Apache Superset ("The Analyst")** — the SQL is submitted to Superset's `/sqllab/execute`
   API. Superset's own engine runs it against DuckDB. This is the same code path as if a human
   analyst typed the query into SQL Lab — it is logged there, and can be opened, verified, and
   turned into a chart by anyone on the team, with zero LLM involvement.
4. **Phi-4-mini-reasoning ("Explainer")** — reads only the *returned result table* (via a
   Markdown bridge, same idea as before) and writes the plain-language answer + business
   implication. It never sees or invents raw data outside of what Superset actually returned.
5. **Self-correction** — if Superset rejects the SQL (syntax error, unknown column, etc.), the
   error message is fed back to Qwen for up to 3 attempts — same defensive retry pattern as
   before, just correcting SQL instead of Python.

A safety guard also rejects anything that isn't a single `SELECT`/`WITH` statement, so the LLM
layer can never write, alter, or drop data inside your BI tool.

**Multi-file folders (the real company use case):** every file in the folder becomes its own table in DuckDB, named after the file (e.g. `orders.csv` -> table `orders`). Nothing gets silently merged. When a question needs more than one file — a join across `orders` and `customers`, or a union across `sales_jan.xlsx` and `sales_feb.xlsx` — Qwen sees every table's schema at once and writes the JOIN/UNION itself; Superset still runs the whole query as one statement, so the join logic is just as auditable as a single-table query.

---


## Step 0 — Superset runs locally, in-process with this notebook (no Docker)

The Docker route means Superset has its own container filesystem, which is a separate
machine from the notebook's point of view — any file path they share has to be explicitly
bind-mounted, or Superset silently ends up looking at an empty database with the same
filename. That's the whole class of bug from before.

The simpler fix: skip Docker. Superset is just a Flask app — `pip install apache-superset`
runs it in a normal local process, on the same filesystem as this notebook, using SQLite as
its own metadata store (fine for single-user local use). Now there is exactly **one**
`DUCKDB_PATH`, used by both sides, no volume mounts, no host/container split.

Cell 2 below installs `apache-superset` alongside the other notebook dependencies, and a new
Cell 3 initializes and launches it as a background process automatically — nothing to run in
a separate terminal. Just make sure port `8088` is free and **Ollama is running**:
```bash
ollama serve   # if not already running as a background service
```
Then run the cells below in order.


In [ ]:
# ── Cell 2: Install dependencies (notebook side only — NOT Superset itself) ───
import subprocess, sys

packages = [
    'gradio', 'ollama', 'pandas', 'openpyxl', 'xlrd',
    'tabulate', 'requests', 'duckdb',
    'apache-superset', 'duckdb-engine', 'flask-cors',
]
for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('\n✅ All packages installed!')


In [ ]:
# ── Cell 3: Pull both Ollama models ────────────────────────────────────────────
import subprocess

REASONING_MODEL = 'phi4-mini-reasoning'         # Plans + Explains only — never touches data directly
CODING_MODEL     = 'qwen2.5-coder:7b-instruct'  # Writes SQL only — never executes it

for model in [REASONING_MODEL, CODING_MODEL]:
    print(f'Pulling {model}...')
    result = subprocess.run(['ollama', 'pull', model], capture_output=False)
    if result.returncode == 0:
        print(f'  ✅ {model} ready!')
    else:
        print(f'  ⚠️  Pull may have failed for {model}')


In [ ]:
# ── Cell 3b: Initialize + launch Superset locally (no Docker) ──────────────────
# Runs Superset as a background process on this same machine, in this same filesystem,
# so the DuckDB file this notebook writes and the one Superset queries are always literally
# the same file -- no container boundary, no path translation, nothing to keep in sync.

import os, sys, subprocess, time, socket
from pathlib import Path as _Path

SUPERSET_HOME = str(_Path.cwd() / '.superset_home')
os.environ['SUPERSET_HOME'] = SUPERSET_HOME
os.environ['FLASK_APP'] = 'superset'
os.makedirs(SUPERSET_HOME, exist_ok=True)

# A fixed secret key avoids Superset refusing to start on repeat runs ("changeme" warning
# becomes an error in recent versions if left as the placeholder).
secret_key_path = _Path(SUPERSET_HOME) / 'secret_key.txt'
if not secret_key_path.exists():
    import secrets
    secret_key_path.write_text(secrets.token_hex(32))
os.environ['SUPERSET_SECRET_KEY'] = secret_key_path.read_text().strip()

config_path = _Path(SUPERSET_HOME) / 'superset_config.py'

# FIX 1: Use forward slashes for the SQLite URI on Windows to prevent SQLAlchemy crashes.
db_path_posix = (_Path(SUPERSET_HOME) / 'superset.db').as_posix()

# FIX 2: We removed the `if not exists:` check so it forcibly overwrites the config 
# on this run, clearing out the bad backslash path from your previous attempt.
config_path.write_text(
    f"SECRET_KEY = {os.environ['SUPERSET_SECRET_KEY']!r}\n"
    f"SQLALCHEMY_DATABASE_URI = 'sqlite:///{db_path_posix}'\n"
)
os.environ['SUPERSET_CONFIG_PATH'] = str(config_path)

def _port_open(port, host='127.0.0.1'):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

SUPERSET_PORT = 8088

# FIX 3: Helper function to explicitly pass the environment and catch/print errors.
import shutil

# FIX 3: Call the 'superset' CLI directly instead of using python -m
def run_superset_cmd(args, check=True):
    # Locate the actual superset executable in your Windows Python/Scripts folder
    superset_exe = shutil.which('superset')
    if not superset_exe:
        raise RuntimeError("❌ Could not find the 'superset' command. Make sure 'apache-superset' installed correctly.")
        
    cmd = [superset_exe] + args
    try:
        return subprocess.run(cmd, check=check, env=os.environ.copy(), capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(f"❌ Command failed: {' '.join(cmd)}")
        print(f"--- STDOUT ---\n{e.stdout}")
        print(f"--- STDERR ---\n{e.stderr}")
        raise

if _port_open(SUPERSET_PORT):
    print(f'✅ Something is already listening on :{SUPERSET_PORT} -- assuming it\'s our Superset '
          f'from a previous run of this notebook. Skipping init/launch.')
else:
    print('🔧 First-time setup: superset db upgrade ...')
    run_superset_cmd(['db', 'upgrade'])

    print('🔧 Creating admin user (ok if it already exists) ...')
    # We set check=False here so it safely ignores errors if the user already exists
    run_superset_cmd([
        'fab', 'create-admin',
        '--username', 'admin',
        '--firstname', 'Admin', '--lastname', 'User',
        '--email', 'admin@example.com',
        '--password', 'admin',
    ], check=False)

    print('🔧 superset init (roles + permissions) ...')
    run_superset_cmd(['init'])

    print(f'🚀 Launching Superset on :{SUPERSET_PORT} in the background ...')
    log_path = _Path(SUPERSET_HOME) / 'superset_server.log'
    log_file = open(log_path, 'a')
    superset_proc = subprocess.Popen(
        [sys.executable, '-m', 'flask', 'run', '-p', str(SUPERSET_PORT),
         '--host', '0.0.0.0', '--without-threads'],
        env=os.environ.copy(), stdout=log_file, stderr=subprocess.STDOUT,
    )

    for _ in range(60):
        if _port_open(SUPERSET_PORT):
            break
        time.sleep(1)
    else:
        raise RuntimeError(
            f'Superset did not come up on :{SUPERSET_PORT} within 60s. '
            f'Check the log for details: {log_path}'
        )
    print(f'✅ Superset is up at http://localhost:{SUPERSET_PORT}  (log: {log_path})')

print('   Login with the SUPERSET_USERNAME / SUPERSET_PASSWORD set in Cell 5 (default admin/admin).')

In [ ]:
# ── Cell 4: Imports ─────────────────────────────────────────────────────────────
import os, re, json, base64, tempfile
from pathlib import Path

import pandas as pd
import duckdb
import requests
import ollama
import gradio as gr

try:
    from tabulate import tabulate as _tabulate
    _HAS_TABULATE = True
except ImportError:
    _HAS_TABULATE = False

print(f'✅ Imports OK  |  Gradio {gr.__version__}')


In [ ]:
# ── Cell 5: Config ──────────────────────────────────────────────────────────────
# Models
REASONING_MODEL = 'phi4-mini-reasoning'
CODING_MODEL    = 'qwen2.5-coder:7b-instruct'
REASONING_CTX   = 8192
CODING_CTX      = 12288

# Superset connection (point this at your running Superset instance)
SUPERSET_URL      = 'http://localhost:8088'
SUPERSET_USERNAME = 'admin'
SUPERSET_PASSWORD = 'admin'          # change to match your instance
SUPERSET_DB_NAME  = 'LLM Analyst (DuckDB)'   # name of the Database connection Superset will use

# Local DuckDB file — the single source of truth both this notebook and Superset read from.
# Superset now runs locally (Cell 3b), in the same filesystem as this notebook, so there's
# no host/container split to manage -- one path, used by both sides.
DUCKDB_PATH = str(Path.cwd() / 'analyst_data.duckdb')

MAX_ATTEMPTS = 3   # self-correction attempts if Superset rejects the generated SQL
SUPPORTED_EXTENSIONS = ('.csv', '.xlsx', '.xlsm', '.xlsb', '.xls', '.tsv')

DATAFRAMES = {}    # filename -> DataFrame, populated by the loaders below

print('✅ Config ready.')
print(f'   Superset   : {SUPERSET_URL}')
print(f'   DuckDB file: {DUCKDB_PATH}')
print(f'   Reasoning  : {REASONING_MODEL}  (ctx={REASONING_CTX})')
print(f'   SQL writer : {CODING_MODEL}  (ctx={CODING_CTX})')


In [ ]:
# ── Cell 6: Data loading + sync into DuckDB ─────────────────────────────────────
# Loading logic is unchanged from the original — the difference is what happens next:
# instead of keeping data only as in-memory DataFrames for a Python sandbox, we persist
# it into a DuckDB file that Superset itself will connect to and query.

def _try_read_csv(path):
    encodings = ['utf-8-sig', 'utf-8', 'cp1252', 'latin1']
    separators = [',', ';', '\t', '|']
    last_err = None
    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False,
                              on_bad_lines='skip', engine='python', sep=None)
            if df.shape[1] > 1 or len(df) > 0:
                return df
        except Exception as e:
            last_err = e
        for sep in separators:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, low_memory=False,
                                  on_bad_lines='skip', engine='c')
                if df.shape[1] > 1 or len(df) > 0:
                    return df
            except Exception as e:
                last_err = e
    raise ValueError(f'Could not parse CSV — last error: {last_err}')


def load_file(tmp_path, orig_name):
    ext = Path(orig_name).suffix.lower()
    try:
        if ext in ('.csv', '.tsv'):
            return _try_read_csv(tmp_path)
        if ext in ('.xlsx', '.xlsm', '.xlsb'):
            engine = 'openpyxl'
        elif ext == '.xls':
            engine = 'xlrd'
        else:
            return None
        sheets = pd.read_excel(tmp_path, sheet_name=None, engine=engine)
        if len(sheets) == 1:
            return next(iter(sheets.values()))
        return sheets
    except Exception as e:
        print(f'  ⚠️  Could not load {orig_name}: {e}')
        return None


def _table_name(fname):
    stem = Path(fname).stem
    return re.sub(r'[^a-zA-Z0-9_]', '_', stem).lower()


def sync_to_duckdb():
    '''Writes every loaded DataFrame into the shared DuckDB file as a real table.
    This is what Superset will actually query — the LLM never sees or touches this file directly.'''
    if not DATAFRAMES:
        print('⚠️  No data loaded yet.')
        return
    con = duckdb.connect(DUCKDB_PATH)
    for fname, df in DATAFRAMES.items():
        table = _table_name(fname)
        con.register('tmp_df', df)
        con.execute(f'CREATE OR REPLACE TABLE "{table}" AS SELECT * FROM tmp_df')
        con.unregister('tmp_df')
        print(f'  ✅ synced `{table}`  ({len(df):,} rows, {len(df.columns)} cols)')
    con.close()
    print(f'\n📦 DuckDB file ready at: {DUCKDB_PATH}')
    print('   → In Superset this is registered as the database:', SUPERSET_DB_NAME)


def load_files(file_objs, source_label='uploaded files'):
    loaded = {}
    for f in file_objs:
        path = f.name if hasattr(f, 'name') else f
        orig_name = os.path.basename(path)
        if not orig_name.lower().endswith(SUPPORTED_EXTENSIONS):
            continue
        df = load_file(path, orig_name)
        if df is None:
            continue
        if isinstance(df, dict):
            for sheet_name, sdf in df.items():
                DATAFRAMES[f'{orig_name}::{sheet_name}'] = sdf
        else:
            DATAFRAMES[orig_name] = df
    if DATAFRAMES:
        sync_to_duckdb()
    return DATAFRAMES, source_label


def load_folder(folder_path):
    import glob
    paths = []
    for ext in SUPPORTED_EXTENSIONS:
        paths.extend(glob.glob(os.path.join(folder_path, f'*{ext}')))
    return load_files(paths, source_label=folder_path)


print('✅ Data loading helpers ready.')


In [ ]:
# ── Cell 7: Superset API client ─────────────────────────────────────────────────
# This class is the entire "hands" that touch real analysis: it logs into Superset and
# submits SQL to Superset's own /sqllab/execute endpoint. Superset's engine runs it —
# not this notebook, not the LLM. Every query submitted here shows up in Superset's own
# SQL Lab query history, exactly as if a human had typed it in.

class SupersetClient:
    def __init__(self, base_url, username, password):
        self.base_url = base_url.rstrip('/')
        self.session = requests.Session()
        self.access_token = None
        self.csrf_token = None
        self._login(username, password)

    def _login(self, username, password):
        r = self.session.post(f'{self.base_url}/api/v1/security/login', json={
            'username': username, 'password': password, 'provider': 'db', 'refresh': True
        })
        r.raise_for_status()
        self.access_token = r.json()['access_token']
        r2 = self.session.get(f'{self.base_url}/api/v1/security/csrf_token/',
                               headers=self._auth_headers())
        r2.raise_for_status()
        self.csrf_token = r2.json()['result']

    def _auth_headers(self):
        return {'Authorization': f'Bearer {self.access_token}'}

    def _write_headers(self):
        h = self._auth_headers()
        h.update({'X-CSRFToken': self.csrf_token, 'Referer': self.base_url,
                   'Content-Type': 'application/json'})
        return h

    def ensure_database(self, name=None, sqlalchemy_uri=None):
        '''Finds (or creates) the Superset Database connection pointing at our DuckDB file.
        If a connection with this name already exists but points at a DIFFERENT uri than the
        one we expect (e.g. left over from an earlier run with a different DUCKDB_PATH),
        it is updated in place -- otherwise Superset keeps querying a stale/empty file forever,
        which looks exactly like "table does not exist" even though your SQL is correct.'''
        name = name or SUPERSET_DB_NAME
        uri = sqlalchemy_uri or f'duckdb:///{DUCKDB_PATH}'
        r = self.session.get(f'{self.base_url}/api/v1/database/?q=(page_size:100)',
                              headers=self._auth_headers())
        r.raise_for_status()
        for row in r.json().get('result', []):
            if row['database_name'] == name:
                existing_uri = row.get('sqlalchemy_uri') or row.get('sqlalchemy_uri_placeholder')
                if existing_uri and existing_uri != uri:
                    upd = self.session.put(
                        f'{self.base_url}/api/v1/database/{row["id"]}',
                        headers=self._write_headers(),
                        json={'database_name': name, 'sqlalchemy_uri': uri},
                    )
                    if upd.status_code >= 400:
                        print(f'  ⚠️  Could not update stale Superset DB URI '
                              f'({existing_uri!r} -> {uri!r}): {upd.text}')
                    else:
                        print(f'  🔄 Updated Superset DB connection URI: {existing_uri!r} -> {uri!r}')
                return row['id']
        payload = {'database_name': name, 'sqlalchemy_uri': uri, 'engine': 'duckdb'}
        r = self.session.post(f'{self.base_url}/api/v1/database/',
                               headers=self._write_headers(), json=payload)
        if r.status_code >= 400:
            raise RuntimeError(
                f'Could not create Superset database connection: {r.text}\n'
                'Did you install duckdb-engine inside the Superset container? See Step 0 above.'
            )
        return r.json()['id']

    def run_sql(self, sql, database_id, schema='main'):
        '''Submits SQL to Superset's SQL Lab execute endpoint — this is the real analysis step.'''
        payload = {'database_id': database_id, 'sql': sql, 'schema': schema,
                   'runAsync': False, 'select_as_cta': False, 'expand_data': True}
        r = self.session.post(f'{self.base_url}/api/v1/sqllab/execute/',
                               headers=self._write_headers(), json=payload)
        if r.status_code != 200:
            try:
                err = r.json().get('message') or r.text
            except Exception:
                err = r.text
            return {'error': str(err)}
        body = r.json()
        if body.get('status') == 'failed' or body.get('error'):
            return {'error': str(body.get('error') or body.get('errors'))}
        return {
            'columns': [c['name'] for c in body.get('columns', [])],
            'data': body.get('data', []),
        }

    def sqllab_url(self):
        return f'{self.base_url}/sqllab/'


print('✅ SupersetClient defined. Connect with: superset = SupersetClient(SUPERSET_URL, SUPERSET_USERNAME, SUPERSET_PASSWORD)')


In [ ]:
# ── Cell 8: Connect to Superset ────────────────────────────────────────────────
try:
    superset = SupersetClient(SUPERSET_URL, SUPERSET_USERNAME, SUPERSET_PASSWORD)
    print(f'✅ Logged into Superset at {SUPERSET_URL}')
except Exception as e:
    superset = None
    print(f'❌ Could not connect to Superset: {e}')
    print('   Make sure Superset is running (see Step 0) and the credentials in Cell 5 are correct.')


In [ ]:
# ── Cell 9: LLM support layer — SQL generation + explanation ──────────────────
#
# MODEL ROUTING (support/explain roles only — Superset does the actual analysis):
#   CODING_MODEL    (qwen2.5-coder)     → translate question -> one read-only SQL query
#   REASONING_MODEL (phi4-mini-reasoning) → explain the result Superset returned

FORBIDDEN_SQL = re.compile(
    r'\b(insert|update|delete|drop|alter|create|attach|detach|copy|pragma|'
    r'export|import|call|install|load|grant|revoke|vacuum|checkpoint)\b',
    re.IGNORECASE,
)

# Phase 2 Strict Constraints: The Superset BI Architect Manual
SUPERSET_SYSTEM_PROMPT = """You are an autonomous Data Analyst Agent operating via Apache Superset, backed by a DuckDB engine. Your primary function is to translate user intent into flawless, execution-ready SQL queries that map perfectly to Superset's visualization capabilities.

I. The Superset Chart Arsenal (Maximize Visual Impact)
Do not limit yourself to basic bar charts. When a user asks for insights or dashboard creation, proactively select the most powerful visualization:
* Big Number KPIs: For single-value metrics with trendline comparisons.
* Time-Series (Advanced): Line charts, Area charts, and Bar charts with rolling averages.
* Geospatial (Deck.gl): Hexagons, Scatterplots, Paths, and Polygons for mapping latitude/longitude.
* Relational & Flow: Sankey Diagrams and Chord Diagrams.
* Hierarchical: Sunburst charts and Treemaps.
* Statistical: Box plots, Violin plots, and Histograms.

II. Strict SQL & Execution Constraints (DuckDB Dialect)
1. Single Result Set per Query: ONE read-only SELECT statement per request. No scripts.
2. No Cartesian Dashboards: DO NOT CROSS JOIN unrelated metrics (e.g. KPIs + Map Coordinates). Generate independent queries.
3. Strict Date/Time Casting: Cast string dates to timestamps (e.g., column::TIMESTAMP) before math.
4. DuckDB Functions Only: Use DATE_TRUNC('month', col) and DATE_DIFF('day', start, end). Do NOT use SQLite functions like julianday().
5. Pre-Aggregate for Performance: Use GROUP BY heavily. Superset wants aggregated chart data, not raw rows."""

def is_safe_sql(sql):
    s = sql.strip().rstrip(';').strip()
    if not re.match(r'^(select|with)\b', s, re.IGNORECASE):
        return False
    if FORBIDDEN_SQL.search(s):
        return False
    if ';' in s:  # no stacked statements
        return False
    return True

def _build_schema():
    if not DATAFRAMES:
        return ''
    lines = []
    for fname, df in DATAFRAMES.items():
        table = _table_name(fname)
        col_info = ', '.join(f'"{c}"({str(dt)})' for c, dt in df.dtypes.items())
        lines.append(f'Table `{table}`: {col_info}')
    return '\n'.join(lines)

def _strip_think(text):
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

def rows_to_markdown(columns, rows, max_rows=30):
    if not columns:
        return '_(no columns returned)_'
    trimmed = rows[:max_rows]
    list_of_lists = [[r.get(c, '') for c in columns] for r in trimmed]
    if _HAS_TABULATE:
        return _tabulate(list_of_lists, headers=columns, tablefmt='pipe')
    header = '| ' + ' | '.join(columns) + ' |'
    sep = '|' + '|'.join(['---'] * len(columns)) + '|'
    body = '\n'.join('| ' + ' | '.join(str(item) for item in row) + ' |' for row in list_of_lists)
    return '\n'.join([header, sep, body])

def generate_sql(question, schema_text, error_context=None):
    system = SUPERSET_SYSTEM_PROMPT + "\n\nOutput EXACTLY one read-only SQL query inside a ```sql block. No prose."
    user = f'### Schema\n{schema_text}\n\n### Question\n{question}'
    if error_context:
        user += (
            f"\n\n### Previous attempt failed\nSQL:\n{error_context['sql']}\n"
            f"Error from Superset:\n{error_context['error']}\nFix the SQL and try again."
        )
    resp = ollama.chat(
        model=CODING_MODEL,
        messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': user}],
        options={'temperature': 0.1, 'num_ctx': CODING_CTX},
    )
    text = resp['message']['content']
    m = re.search(r'```sql\s*(.*?)```', text, re.DOTALL | re.IGNORECASE)
    sql = m.group(1).strip() if m else text.strip()
    return sql.rstrip(';').strip()

def explain_results(question, sql, markdown_table):
    system = (
        'You are a business analyst. You are given SQL that was already executed and verified '
        'by Apache Superset (a deterministic BI tool) and the exact result it returned as '
        'a Markdown table. Using ONLY the numbers in that table, write:\n'
        '1. **Answer** — one direct sentence.\n'
        '2. **Analysis** — brief interpretation of the numbers.\n'
        '3. **Business Implication** — one short paragraph.\n'
        'Never invent a number that is not present in the table.'
    )
    user = f'Question: {question}\n\nSQL executed by Superset:\n```sql\n{sql}\n```\n\nResult:\n{markdown_table}'
    resp = ollama.chat(
        model=REASONING_MODEL,
        messages=[{'role': 'system', 'content': system},
                  {'role': 'user', 'content': f'<think>\n{user}\n</think>'}],
        options={'temperature': 0.3, 'num_ctx': REASONING_CTX},
    )
    return _strip_think(resp['message']['content'])

def ask_analyst(question, max_attempts=MAX_ATTEMPTS):
    '''Returns (explanation_text, sql_used, markdown_table_or_None).'''
    if superset is None:
        return '❌ Not connected to Superset. Re-run Cell 8.', None, None
    schema_text = _build_schema()
    if not schema_text:
        return '⚠️ No data loaded yet — load a file first.', None, None
    try:
        db_id = superset.ensure_database()
    except Exception as e:
        return f'❌ {e}', None, None

    error_context, last_sql = None, None
    for attempt in range(1, max_attempts + 1):
        sql = generate_sql(question, schema_text, error_context)
        last_sql = sql
        if not is_safe_sql(sql):
            error_context = {'sql': sql, 'error': 'Rejected by safety guard: only a single read-only SELECT/WITH statement is allowed.'}
            continue
        result = superset.run_sql(sql, db_id)
        if result.get('error'):
            error_context = {'sql': sql, 'error': result['error']}
            continue
        md_table = rows_to_markdown(result['columns'], result['data'])
        explanation = explain_results(question, sql, md_table)
        return explanation, sql, md_table

    err = error_context['error'] if error_context else 'unknown error'
    return (f'❌ Superset could not run a valid query for this after {max_attempts} attempts.\n\n'
            f'**Last SQL tried:**\n```sql\n{last_sql}\n```\n**Last error:** {err}\n\n'
            f'You can debug this directly in Superset SQL Lab: {superset.sqllab_url()}'), last_sql, None

# Phase 1 Auto-Bootstrapping Pipeline
def bootstrap_overview():
    '''Generates 3 foundational queries to build a Data Overview Dashboard upon data upload.'''
    if superset is None or not DATAFRAMES:
        return "⚠️ Cannot generate overview.", "", ""
    schema_text = _build_schema()
    try:
        db_id = superset.ensure_database()
    except Exception:
        return "⚠️ Database error.", "", ""

    prompt = (
        f"### Schema\n{schema_text}\n\n"
        "Generate exactly 3 distinct SQL queries to build a 'Data Overview Dashboard' for this new dataset. "
        "Include: 1) A high-level aggregate KPI, 2) A categorical breakdown/distribution, 3) A time-series trend or top 10 list. "
        "Output each query in its own separate ```sql block. Ensure DuckDB compatibility."
    )

    resp = ollama.chat(
        model=CODING_MODEL,
        messages=[{'role': 'system', 'content': SUPERSET_SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}],
        options={'temperature': 0.2, 'num_ctx': CODING_CTX},
    )
    
    sql_blocks = re.findall(r'```sql\s*(.*?)\s*```', resp['message']['content'], re.DOTALL | re.IGNORECASE)
    if not sql_blocks:
        return "Could not automatically generate overview queries.", "", ""

    results_md, valid_sqls = [], []
    for sql in sql_blocks[:3]:
        sql = sql.strip().rstrip(';')
        if not is_safe_sql(sql): continue
        res = superset.run_sql(sql, db_id)
        if res.get('error'): continue
        valid_sqls.append(sql)
        results_md.append(f"**Query Result:**\n{rows_to_markdown(res['columns'], res['data'])}")

    if not valid_sqls:
        return "Auto-generated overview queries failed execution.", "", ""

    combined_sqls = "\n\n".join([f"-- Overview Chart {i+1}\n{s}" for i, s in enumerate(valid_sqls)])
    combined_results = "\n\n".join(results_md)

    explain_prompt = (
        "You are a BI Analyst. Here are the results of foundational queries executed to create a Data Overview Dashboard. "
        "Write a cohesive, highly engaging summary (2-3 paragraphs) giving the user a general overview of their uploaded data. "
        "Highlight key metrics, scale, and interesting patterns. Do not invent data."
    )
    exp_resp = ollama.chat(
        model=REASONING_MODEL,
        messages=[{'role': 'system', 'content': explain_prompt}, {'role': 'user', 'content': f'<think>\n{combined_results}\n</think>'}],
        options={'temperature': 0.3, 'num_ctx': REASONING_CTX},
    )
    
    return _strip_think(exp_resp['message']['content']), combined_sqls, combined_results

print('✅ Support/Explain pipeline ready. Phase 1 Bootstrap & Phase 2 Constraints injected.')

In [ ]:
# ── Cell 10: Gradio UI ──────────────────────────────────────────────────────────

CSS = '''
#app-title {
  background: linear-gradient(90deg, #6d28d9, #0369a1);
  -webkit-background-clip: text;
  -webkit-text-fill-color: transparent;
  font-weight: 700;
  text-align: center;
}
#chat-log { min-height: 550px; max-height: 650px; overflow-y: auto; }
#sql-box textarea { font-family: monospace; }
'''

def get_table_summary():
    if not DATAFRAMES:
        return '_No data loaded yet._'
    rows = ['| Table (in DuckDB / Superset) | Rows | Cols |', '|---|---|---|']
    for name in DATAFRAMES:
        df = DATAFRAMES[name]
        rows.append(f'| `{_table_name(name)}` | {len(df):,} | {len(df.columns)} |')
    return '\n'.join(rows)

def render_log(history):
    if not history:
        return '_Ask a question to get started._'
    blocks = []
    for q, a in history:
        blocks.append(f'**🙋 You:** {q}\n\n**🤖 Analyst:** {a}')
    return '\n\n---\n\n'.join(blocks)

# Updated Loaders to trigger Phase 1 Bootstrap
def on_load_uploaded(uploaded_files, history):
    if not uploaded_files:
        return '⚠️ Select files first.', history, render_log(history), '', ''
    if not isinstance(uploaded_files, list):
        uploaded_files = [uploaded_files]
    dfs, _ = load_files(uploaded_files, source_label='uploaded folder')
    if dfs:
        status = f'✅ **Ready!** Loaded {len(dfs)} table(s) from the folder and synced to DuckDB.'
        overview, sqls, tables = bootstrap_overview()
        history = list(history) + [("*(Pipeline Action: Auto-Generated Data Overview Dashboard)*", overview)]
        return status, history, render_log(history), sqls, tables
    return '❌ Failed to load files. Check format.', history, render_log(history), '', ''

def on_load_folder_path(folder_path, history):
    if not folder_path or not folder_path.strip():
        return '⚠️ Enter a folder path first.', history, render_log(history), '', ''
    dfs, _ = load_folder(folder_path.strip())
    if dfs:
        status = f'✅ **Ready!** Loaded {len(dfs)} table(s) from the folder and synced to DuckDB.'
        overview, sqls, tables = bootstrap_overview()
        history = list(history) + [("*(Pipeline Action: Auto-Generated Data Overview Dashboard)*", overview)]
        return status, history, render_log(history), sqls, tables
    return '❌ Failed to load files. Check path.', history, render_log(history), '', ''

def respond(message, history):
    if not message or not message.strip():
        return render_log(history), history, '', ''
    explanation, sql, md_table = ask_analyst(message)
    history = list(history) + [(message, explanation)]
    sql_display = f'```sql\n{sql}\n```' if sql else '_(no query generated)_'
    table_display = md_table or '_(no result table)_'
    return render_log(history), history, sql_display, table_display

with gr.Blocks(title='Autonomous Data Analyst — BI-Backed', css=CSS) as demo:
    gr.Markdown(
        "<h1 id='app-title'>✦ Autonomous Data Analyst — BI-Backed</h1>"
        f"<p style='text-align:center'>Analysis by <b>Apache Superset</b> · Explained by "
        f"<code>{REASONING_MODEL}</code> + <code>{CODING_MODEL}</code></p>"
    )

    with gr.Accordion('📂 Attach Data', open=True):
        with gr.Row():
            upload = gr.File(label='Upload Folder (all files at once)', file_count='directory',
                              type='filepath', height=100)
            folder_path = gr.Textbox(label='Or enter local folder path (server-side)',
                                      placeholder='/path/to/data')
        with gr.Row():
            load_upload_btn = gr.Button('⚡ Load Uploaded', variant='primary')
            load_path_btn = gr.Button('⚡ Load Path')
        status = gr.Markdown('_No data loaded yet._')

    chat_state = gr.State([]) 

    with gr.Row():
        with gr.Column(scale=2):
            chat_log = gr.Markdown('_Ask a question to get started._', elem_id='chat-log')
            msg = gr.Textbox(label='', placeholder='Ask about your data...')
            with gr.Row():
                send_btn = gr.Button('Send', variant='primary')
                clear_btn = gr.Button('Clear')
        with gr.Column(scale=1):
            gr.Markdown(f'🔗 **Open in Superset SQL Lab:** {SUPERSET_URL}/sqllab/')
            tables_md = gr.Markdown(label='Loaded tables')
            sql_out = gr.Markdown(label='SQL executed by Superset', elem_id='sql-box')
            table_out = gr.Markdown(label='Result returned by Superset')

    # Handlers now push Chat State and auto-populate the overview
    load_upload_btn.click(
        on_load_uploaded, 
        [upload, chat_state], 
        [status, chat_state, chat_log, sql_out, table_out]
    ).then(get_table_summary, None, tables_md)
    
    load_path_btn.click(
        on_load_folder_path, 
        [folder_path, chat_state], 
        [status, chat_state, chat_log, sql_out, table_out]
    ).then(get_table_summary, None, tables_md)

    send_btn.click(respond, [msg, chat_state], [chat_log, chat_state, sql_out, table_out]).then(lambda: '', None, msg)
    msg.submit(respond, [msg, chat_state], [chat_log, chat_state, sql_out, table_out]).then(lambda: '', None, msg)
    clear_btn.click(lambda: ('_Ask a question to get started._', [], '', ''), None,
                     [chat_log, chat_state, sql_out, table_out])

print('✅ UI defined (Proactive Overview enabled).')

In [ ]:
# ── Cell 11: Launch ─────────────────────────────────────────────────────────────
try:
    listed = ollama.list().get('models', [])
    model_names = [m.get('name') or m.get('model', '') for m in listed]
    for model in [REASONING_MODEL, CODING_MODEL]:
        found = any(model in m for m in model_names)
        status_str = '✅' if found else '⚠️  NOT FOUND — run: ollama pull ' + model
        print(f'{status_str}  {model}')
except Exception as e:
    print(f'❌ Cannot reach Ollama: {e}')
    print('   Run `ollama serve` in a terminal first.')

if superset is None:
    print('❌ Not connected to Superset — fix Cell 8 before continuing.')

# Kill any server this kernel already has running (e.g. from a previous
# run of this notebook) so edits actually take effect instead of an old
# process silently continuing to serve stale code.
try:
    demo.close()
    print('🔌 Closed a previously running server in this kernel.')
except Exception:
    pass

launched = False
for port in range(7860, 7871):
    try:
        print(f'\n🚀 Launching on port {port}...')
        demo.launch(server_name='0.0.0.0', server_port=port, share=False, inbrowser=True)
        launched = True
        break
    except OSError:
        print(f'  Port {port} busy, trying next...')

if not launched:
    print('❌ No free port in range 7860-7870.')
